# Tokenization & Answer Span Alignment - LAB 10

Notebook kiểm tra từng bước: input của QA model, offset mapping, sliding window và ánh xạ character span sang token span. Hàm alignment dùng chung với phần fine-tuning nằm trong qa_model.py (char_span_to_token_span, prepare_features).

In [1]:
import inspect
import warnings
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from datasets import load_dataset
import qa_model as qm

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
question = "What is the capital of France?"
context = "The capital of France is Paris."


## 1. Input của QA: input_ids, attention_mask, token_type_ids

So sánh 3 tokenizer. Không phải model nào cũng dùng token_type_ids.

In [2]:
for name in ["bert-base-uncased", "distilbert-base-uncased", "roberta-base"]:
    t = AutoTokenizer.from_pretrained(name)
    enc = t(question, context)
    print("=" * 20, name)
    print("keys          :", list(enc.keys()))
    print("tokens        :", t.convert_ids_to_tokens(enc["input_ids"]))
    print("input_ids     :", enc["input_ids"])
    print("attention_mask:", enc["attention_mask"])
    print("token_type_ids:", enc.get("token_type_ids", "KHÔNG CÓ"))
    print("sequence_ids  :", enc.sequence_ids())

==================== bert-base-uncased
keys          : ['input_ids', 'token_type_ids', 'attention_mask']
tokens        : ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', '[SEP]']
input_ids     : [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
sequence_ids  : [None, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, None]


==================== distilbert-base-uncased
keys          : ['input_ids', 'token_type_ids', 'attention_mask']
tokens        : ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', '[SEP]']
input_ids     : [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
sequence_ids  : [None, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, None]


==================== roberta-base
keys          : ['input_ids', 'attention_mask']
tokens        : ['<s>', 'What', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof', 'ĠFrance', '?', '</s>', '</s>', 'The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis', 'ĠParis', '.', '</s>']
input_ids     : [0, 2264, 16, 5, 812, 9, 1470, 116, 2, 2, 133, 812, 9, 1470, 16, 2201, 4, 2]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids: KHÔNG CÓ
sequence_ids  : [None, 0, 0, 0, 0, 0, 0, 0, None, None, 1, 1, 1, 1, 1, 1, 1, None]


In [3]:
# Tokenizer DistilBERT trả về token_type_ids nhưng model DistilBERT không nhận tham số này
from transformers import DistilBertForQuestionAnswering, BertForQuestionAnswering
print("DistilBERT forward có token_type_ids:",
      "token_type_ids" in inspect.signature(DistilBertForQuestionAnswering.forward).parameters)
print("BERT forward có token_type_ids      :",
      "token_type_ids" in inspect.signature(BertForQuestionAnswering.forward).parameters)

DistilBERT forward có token_type_ids: False
BERT forward có token_type_ids      : True


Nhận xét:

- BERT: token_type_ids = 0 cho [CLS] question [SEP], = 1 cho context [SEP].
- DistilBERT: tokenizer vẫn sinh token_type_ids nhưng model không có segment embedding, vì vậy collate function trong qa_model.py chỉ đưa input_ids và attention_mask vào model.
- RoBERTa: không có token_type_ids, dùng `<s> q </s></s> c </s>`.
- Cách an toàn để biết token nào thuộc context là sequence_ids() (1 = context).

## 2. Character index vs word index vs token index

In [4]:
answer = "Paris"
a_start = context.index(answer)
a_end = a_start + len(answer)
print("character span:", a_start, "->", a_end - 1, "(inclusive), python slice", (a_start, a_end), repr(context[a_start:a_end]))
print("word index    :", context.split().index("Paris."), "(word = 'Paris.' khi tách theo khoảng trắng)")

enc = tok(question, context, return_offsets_mapping=True)
seq = enc.sequence_ids()
for i, (t, o, s) in enumerate(zip(tok.convert_ids_to_tokens(enc["input_ids"]), enc["offset_mapping"], seq)):
    print(f"{i:>2} {t:<10} offset={o} seq_id={s}")
print("token span    :", qm.char_span_to_token_span(enc["offset_mapping"], seq, a_start, a_end))

character span: 25 -> 29 (inclusive), python slice (25, 30) 'Paris'
word index    : 5 (word = 'Paris.' khi tách theo khoảng trắng)
 0 [CLS]      offset=(0, 0) seq_id=None
 1 what       offset=(0, 4) seq_id=0
 2 is         offset=(5, 7) seq_id=0
 3 the        offset=(8, 11) seq_id=0
 4 capital    offset=(12, 19) seq_id=0
 5 of         offset=(20, 22) seq_id=0
 6 france     offset=(23, 29) seq_id=0
 7 ?          offset=(29, 30) seq_id=0
 8 [SEP]      offset=(0, 0) seq_id=None
 9 the        offset=(0, 3) seq_id=1
10 capital    offset=(4, 11) seq_id=1
11 of         offset=(12, 14) seq_id=1
12 france     offset=(15, 21) seq_id=1
13 is         offset=(22, 24) seq_id=1
14 paris      offset=(25, 30) seq_id=1
15 .          offset=(30, 31) seq_id=1
16 [SEP]      offset=(0, 0) seq_id=None
token span    : (14, 14)


Cùng một answer "Paris": character index 25, word index 5, token index 14. Offset (0, 0) là special token; offset của question và context đều tính lại từ 0 nên phải dùng sequence_ids để chỉ xét token context.

## 3. Core implementation: tokenizer với sliding window

Lấy một context dài của SQuAD để thấy overflow.

In [5]:
ds = load_dataset("rajpurkar/squad")
train = ds["train"]
lens = [len(tok(c)["input_ids"]) for c in train.select(range(3000))["context"]]
idx = max(range(3000), key=lambda i: lens[i])
ex = train[idx]
print("context tokens:", lens[idx])
print("question:", ex["question"])
print("answer  :", ex["answers"])

context tokens: 507
question: When were the Rinbung princes overthrown?
answer  : {'text': ['1565'], 'answer_start': [3]}


In [6]:
inputs = tok(
    ex["question"],
    ex["context"],
    truncation="only_second",
    max_length=384,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)
print("keys:", list(inputs.keys()))
print("số window:", len(inputs["input_ids"]))
print("overflow_to_sample_mapping:", inputs["overflow_to_sample_mapping"])
for w in range(len(inputs["input_ids"])):
    seq = inputs.sequence_ids(w)
    ctx = [i for i, s in enumerate(seq) if s == 1]
    o = inputs["offset_mapping"][w]
    print(f"window {w}: len={len(inputs['input_ids'][w])} attention_mask sum={sum(inputs['attention_mask'][w])} "
          f"context tokens {ctx[0]}..{ctx[-1]} -> chars {o[ctx[0]][0]}..{o[ctx[-1]][1]}")

keys: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping']
số window: 2
overflow_to_sample_mapping: [0, 0]
window 0: len=384 attention_mask sum=384 context tokens 12..382 -> chars 0..1514
window 1: len=275 attention_mask sum=275 context tokens 12..273 -> chars 1027..2099


In [7]:
# Overlap giữa window 0 và window 1: 128 token cuối của window 0 = 128 token context đầu của window 1
w0 = [i for i, s in enumerate(inputs.sequence_ids(0)) if s == 1]
w1 = [i for i, s in enumerate(inputs.sequence_ids(1)) if s == 1]
tail0 = [inputs["input_ids"][0][i] for i in w0[-128:]]
head1 = [inputs["input_ids"][1][i] for i in w1[:128]]
print("128 token cuối window 0 == 128 token đầu window 1:", tail0 == head1)
print("window 0 tail:", tok.decode(tail0[-15:]))
print("window 1 head:", tok.decode(head1[:15]))

128 token cuối window 0 == 128 token đầu window 1: True
window 0 tail: latter accepted the six - year - old lozang gyatso as
window 1 head: 1618, only two years after yonten gyatso died, the


Giải thích:

- input_ids: id của [CLS] question [SEP] context-window [SEP]; question được lặp lại ở mọi window.
- attention_mask: 1 cho token thật, 0 cho padding. Ở đây không padding nên toàn 1; khi train, collate function pad động và mask = 0 ở phần pad.
- offset_mapping: (start_char, end_char) của token trong chuỗi gốc (question hoặc context), special token là (0, 0).
- overflow_to_sample_mapping: window thứ k thuộc example nào. Một example dài sinh nhiều feature, khi post-processing phải gom các window về cùng example để chọn span tốt nhất.
- truncation="only_second": chỉ cắt context, không bao giờ cắt question. stride=128: số token overlap giữa hai window liên tiếp, để answer nằm gần ranh giới vẫn xuất hiện trọn vẹn trong ít nhất một window.

## 4. Answer span alignment

In [8]:
print(inspect.getsource(qm.char_span_to_token_span))

def char_span_to_token_span(offsets, sequence_ids, answer_start, answer_end):
    """
    offsets: list of (start_char, end_char) for every token in one feature.
    sequence_ids: 0 = question, 1 = context, None = special token.
    answer_start: index of first char of the answer in context.
    answer_end: index right after the last char (exclusive).
    Return (start_token, end_token) or None if the answer is not fully inside
    the context part of this window.
    """
    ctx = [i for i, s in enumerate(sequence_ids) if s == 1]
    if not ctx:
        return None
    win_start, win_end = offsets[ctx[0]][0], offsets[ctx[-1]][1]
    # answer truncated / outside this window
    if answer_start < win_start or answer_end > win_end:
        return None

    start_token, end_token = None, None
    for token_idx in ctx:
        start, end = offsets[token_idx]
        # start <= answer_start < end : answer starts inside this token
        # (also covers answer starting in the middle of a tok

In [9]:
a_start = ex["answers"]["answer_start"][0]
a_end = a_start + len(ex["answers"]["text"][0])
for w in range(len(inputs["input_ids"])):
    span = qm.char_span_to_token_span(inputs["offset_mapping"][w], inputs.sequence_ids(w), a_start, a_end)
    if span is None:
        print(f"window {w}: answer nằm ngoài window -> label (0, 0) = [CLS]")
    else:
        s, e = span
        o = inputs["offset_mapping"][w]
        print(f"window {w}: start_token={s} end_token={e} -> {ex['context'][o[s][0]:o[e][1]]!r}")

window 0: start_token=13 end_token=14 -> '1565'
window 1: answer nằm ngoài window -> label (0, 0) = [CLS]


### 4.1 Answer bắt đầu / kết thúc giữa token

In [10]:
def show(question, context, answer_text, answer_start, max_length=384, stride=0, overflow=False):
    enc = tok(question, context, truncation="only_second", max_length=max_length, stride=stride,
              return_overflowing_tokens=overflow, return_offsets_mapping=True)
    n = len(enc["input_ids"]) if overflow else 1
    a_end = answer_start + len(answer_text)
    for w in range(n):
        offs = enc["offset_mapping"][w] if overflow else enc["offset_mapping"]
        seq = enc.sequence_ids(w)
        ids = enc["input_ids"][w] if overflow else enc["input_ids"]
        span = qm.char_span_to_token_span(offs, seq, answer_start, a_end)
        cidx = [i for i, x in enumerate(seq) if x == 1]
        rng = (offs[cidx[0]][0], offs[cidx[-1]][1])
        if span is None:
            print(f"  window {w}: context chars {rng}, answer chars {(answer_start, a_end)} -> None -> label [CLS] (0, 0)")
        else:
            s, e = span
            print(f"  window {w}: tokens {tok.convert_ids_to_tokens(ids[s:e+1])} span=({s},{e}) "
                  f"-> text {context[offs[s][0]:offs[e][1]]!r} | gold {answer_text!r}")

c1 = "The process is called photosynthesis in plants."
print("Bắt đầu giữa token:", tok.tokenize("photosynthesis"))
show("What is synthesis?", c1, "synthesis", c1.index("synthesis"))
c2 = "Roberson left the group in 2000."
print("Kết thúc giữa token:", tok.tokenize("Roberson"))
show("Who left?", c2, "Rober", c2.index("Rober"))

Bắt đầu giữa token: ['photos', '##yn', '##thesis']
  window 0: tokens ['photos', '##yn', '##thesis'] span=(10,12) -> text 'photosynthesis' | gold 'synthesis'
Kết thúc giữa token: ['robe', '##rson']
  window 0: tokens ['robe', '##rson'] span=(5,6) -> text 'Roberson' | gold 'Rober'


Khi answer bắt đầu hoặc kết thúc giữa một subword, điều kiện start <= answer_start < end (và start < answer_end <= end) vẫn chọn được token chứa ký tự đó, nhưng span khôi phục lại từ offset sẽ rộng hơn gold (ví dụ "photosynthesis" thay vì "synthesis"). Đây là giới hạn của span theo token: model không thể dự đoán chính xác gold này, EM tối đa sẽ < 100%.

### 4.2 Answer bị truncation và answer nằm ngoài window

In [11]:
long_ctx = ("Filler sentence number one is here. " * 12) + "The treaty was signed in 1648 in Westphalia."
q = "When was the treaty signed?"
ans = "1648"
st = long_ctx.index(ans)
print("Truncation, không overflow, max_length=64:")
show(q, long_ctx, ans, st, max_length=64)
print("Sliding window max_length=64, stride=16:")
show(q, long_ctx, ans, st, max_length=64, stride=16, overflow=True)

Truncation, không overflow, max_length=64:
  window 0: context chars (0, 250), answer chars (457, 461) -> None -> label [CLS] (0, 0)
Sliding window max_length=64, stride=16:
  window 0: context chars (0, 250), answer chars (457, 461) -> None -> label [CLS] (0, 0)
  window 1: context chars (178, 425), answer chars (457, 461) -> None -> label [CLS] (0, 0)
  window 2: tokens ['1648'] span=(31,31) -> text '1648' | gold '1648'


In [12]:
# Answer nằm vắt qua ranh giới window: window trước chứa một phần answer -> None, window sau chứa trọn -> có span
ctx3 = ("Filler sentence number one is here. " * 6) + "The founder was Sir Isaac Newton of Woolsthorpe Manor in England."
ans = "Sir Isaac Newton of Woolsthorpe Manor"
st = ctx3.index(ans)
show("Who was the founder?", ctx3, ans, st, max_length=64, stride=24, overflow=True)

  window 0: context chars (0, 256), answer chars (232, 269) -> None -> label [CLS] (0, 0)
  window 1: tokens ['sir', 'isaac', 'newton', 'of', 'wool', '##st', '##horpe', 'manor'] span=(26,33) -> text 'Sir Isaac Newton of Woolsthorpe Manor' | gold 'Sir Isaac Newton of Woolsthorpe Manor'


- answer bị truncation (không dùng overflow): answer không nằm trong context đã cắt, hàm trả None, feature được gán label (0, 0) trỏ về [CLS]. Nếu không kiểm tra mà vẫn tìm token theo offset thì sẽ gán nhãn sai.
- answer nằm ngoài window: các window không chứa trọn answer cũng được gán [CLS]. Answer chỉ nằm một phần trong window cũng bị coi là ngoài window, nhờ stride nên vẫn có window khác chứa trọn answer.

## 5. Kiểm tra alignment trên 3.000 câu train

In [13]:
sub = train.shuffle(seed=42).select(range(3000))
exs = {e["id"]: e for e in sub}
for L, S in [(128, 42), (256, 85), (384, 128)]:
    f = qm.build_features(sub, tok, L, S)
    n_in = sum(f["answer_in_window"])
    exact, mism = 0, []
    for x in f:
        if not x["answer_in_window"]:
            continue
        o = x["offset_mapping"]
        e = exs[x["example_id"]]
        rec = e["context"][o[x["start_positions"]][0]:o[x["end_positions"]][1]]
        if rec == e["answers"]["text"][0]:
            exact += 1
        else:
            mism.append((rec, e["answers"]["text"][0]))
    covered = len({x["example_id"] for x in f if x["answer_in_window"]})
    print(f"max_length={L} stride={S}: features={len(f)}, có answer={n_in}, [CLS]={len(f)-n_in}, "
          f"example có ít nhất 1 window chứa answer={covered}/3000, khôi phục đúng={exact}/{n_in}")
print("Ví dụ không khôi phục đúng (answer cắt giữa token / khoảng trắng):")
for m in mism[:8]:
    print("  reconstructed", repr(m[0]), "| gold", repr(m[1]))

max_length=128 stride=42: features=6425, có answer=3602, [CLS]=2823, example có ít nhất 1 window chứa answer=3000/3000, khôi phục đúng=3593/3602


max_length=256 stride=85: features=3330, có answer=3070, [CLS]=260, example có ít nhất 1 window chứa answer=3000/3000, khôi phục đúng=3062/3070


max_length=384 stride=128: features=3030, có answer=3008, [CLS]=22, example có ít nhất 1 window chứa answer=3000/3000, khôi phục đúng=3000/3008
Ví dụ không khôi phục đúng (answer cắt giữa token / khoảng trắng):
  reconstructed 'sex. Men did not show any sexual arousal to non-human visual stimuli,' | gold 'x. Men did not show any sexual arousal to non-human visual stimuli,'
  reconstructed 'early science-fiction and adventure' | gold 'y science-fiction and adventure'
  reconstructed '2010' | gold '1'
  reconstructed '£250 million' | gold '250 million'
  reconstructed '2008' | gold '2'
  reconstructed 'iPhone' | gold 'one'
  reconstructed 'been' | gold 'b'
  reconstructed 'Architecture' | gold 'Architect'


Kết luận: alignment khôi phục đúng gần như toàn bộ answer (3000/3008 ở max_length=384). Các trường hợp lệch là do gold answer cắt giữa từ (ví dụ "Architect" trong "Architecture", "1" trong "2010") hoặc dính ký tự khác ("£250 million"), không phải lỗi của code alignment. Với max_length nhỏ, số feature có nhãn [CLS] tăng mạnh vì nhiều window không chứa answer.